# KoELECTRA 문자 위험 분류 — baseline v2

사진 분석 서비스의 **한국어 텍스트 분류기**를 학습합니다. 전체 파인튜닝과 FP16 혼합 정밀도 학습을 사용하며, PEFT·양자화·OCR·번역·Qwen 호출은 이 학습 범위에 포함되지 않습니다.

기존 `koelectra-baseline-v1`은 보존합니다. 이번 실험은 라벨 충돌 검사와 템플릿 그룹 분리를 적용하므로 이전 무작위 분할의 100% 결과와 직접 비교하지 마세요.

정상 데이터가 합성 택배 안내에 치우치고 위험 라벨에 광고성 스팸이 섞여 있는 한계는 여전히 남아 있습니다. 이 노트북 정리만으로 새 문자에서의 성능 개선이 보장되지는 않습니다.

**실행:** 설치 셀을 한 번 실행 → 세션 재시작 → 설치 셀을 건너뛰고 순서대로 실행 → `학습 노트북 종료`에서 멈추세요. 그 아래 이전 코드는 보존만 했으며 이번 학습 흐름에 포함되지 않습니다.

수정본의 과거 출력은 제거했습니다. 로컬에서는 구조·문법과 데이터 처리 일부를 확인했으며, 실제 학습은 Colab GPU와 Google Drive에서 다시 실행해야 합니다.


## 1. 환경 준비
처음 한 번만 실행하고 세션을 재시작하세요. PyTorch·CUDA는 Colab 제공 환경을 사용합니다.


In [1]:
# 기존 충돌 패키지 정리: 이번 텍스트 분류 실험에서는 사용하지 않습니다.
%pip uninstall -y gradio diffusers transformers tokenizers huggingface-hub
%pip install --no-cache-dir "pandas==2.2.3" "Pillow==11.3.0" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate==1.14.0" "datasets==4.0.0" "scikit-learn==1.6.1" "jedi>=0.16"
%pip check


Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.0 MB 3.4 MB/s eta 0:00:04
   ---------------------- ----------------- 6.8/12.0 MB 23.3 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 30.1 MB/s  0:00:00
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 566.4/566.4 kB ?  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 22.8 MB/s  0:00:00

   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ------------- -------------------------- 1/3 [tokenizers]
   -------------------------- ------------- 2/3 [transformers]
   -------------------------- ------------- 2/3 [transformers]
   --------------------------


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


## 2. 재시작 후 환경 확인 및 실험 설정
이 셀부터 실행하세요. 실험마다 새 폴더를 만들며, 기존 모델을 덮어쓰지 않습니다.


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU 사용 가능:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU 사용 가능: True


In [5]:
import hashlib
import json
import re
import unicodedata
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import huggingface_hub
from transformers import set_seed

assert transformers.__version__ == "4.57.1", (
    "설치 셀 실행 후 커널을 재시작하세요."
)
assert huggingface_hub.__version__ == "0.36.2", (
    "설치 셀 실행 후 커널을 재시작하세요."
)

# 현재 연결된 런타임의 GPU 사용
# 로컬 런타임이면 내 PC의 GPU를 사용합니다.
assert torch.cuda.is_available(), (
    "현재 런타임에서 GPU를 인식하지 못했습니다. "
    "로컬 환경의 PyTorch CUDA 설치를 확인하세요."
)

DEVICE = torch.device("cuda:0")
print("사용 GPU:", torch.cuda.get_device_name(DEVICE), flush=True)

SEED = 42
MODEL_ID = "monologg/koelectra-base-v3-discriminator"
DATASET_ID = "meal-bbang/Korean_message"
MAX_LENGTH = 256

ID2LABEL = {0: "NORMAL", 1: "RISK"}
LABEL_MAP = {2: 1, 3: 0}

# 현재 실행 중인 PC의 사용자 폴더에 저장
# 예: C:/Users/Donggeon/smishing-checker
PROJECT_DIR = Path.home() / "smishing-checker"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = (
    "koelectra-baseline-v2-"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)

set_seed(SEED)

environment = {
    name: version(name)
    for name in [
        "torch", "transformers", "huggingface-hub",
        "tokenizers", "accelerate", "datasets",
        "scikit-learn", "pandas", "numpy", "Pillow",
    ]
}

print(environment)
print("저장 경로:", PROJECT_DIR)
print("실험:", RUN_ID)

사용 GPU: NVIDIA GeForce RTX 5060 Ti
{'torch': '2.11.0+cu128', 'transformers': '4.57.1', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2', 'accelerate': '1.14.0', 'datasets': '4.0.0', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'numpy': '2.5.2', 'Pillow': '11.3.0'}
저장 경로: C:\Users\SSAFY\smishing-checker
실험: koelectra-baseline-v2-20260907T015430973273Z


In [7]:
from google.colab import drive
drive.mount("/content/drive")
#원격 GPU 사용할때
RUN_DIR = PROJECT_DIR / "experiments" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
SPLIT_DIR = RUN_DIR / "splits"
SPLIT_DIR.mkdir()
save_dir = RUN_DIR / "model"
print("실험 저장 경로:", RUN_DIR)


ModuleNotFoundError: No module named 'google'

In [8]:
from pathlib import Path

# 내 PC 사용자 폴더에 프로젝트 저장
PROJECT_DIR = Path.home() / "smishing-checker"

RUN_DIR = PROJECT_DIR / "experiments" / RUN_ID
SPLIT_DIR = RUN_DIR / "splits"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MODEL_DIR = RUN_DIR / "model"

for directory in [
    PROJECT_DIR,
    RUN_DIR,
    SPLIT_DIR,
    CHECKPOINT_DIR,
    MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("프로젝트 경로:", PROJECT_DIR)
print("실험 결과 경로:", RUN_DIR)

프로젝트 경로: C:\Users\SSAFY\smishing-checker
실험 결과 경로: C:\Users\SSAFY\smishing-checker\experiments\koelectra-baseline-v2-20260907T015430973273Z


## 3. 데이터 로드 및 출처 기록
원본 class 1은 제외하고 class 2 → 위험(1), class 3 → 정상(0)으로 매핑합니다. 광고성 스팸과 스미싱이 섞인 임시 라벨 정의입니다.
모델·데이터 저장소의 현재 커밋을 기록하고 그 버전으로 로드합니다.


In [9]:
from datasets import load_dataset
from huggingface_hub import HfApi

hub = HfApi()
DATASET_REVISION = hub.dataset_info(DATASET_ID).sha
MODEL_REVISION = hub.model_info(MODEL_ID).sha
dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION)
raw_df = dataset["train"].to_pandas()
assert {"content", "class"}.issubset(raw_df.columns)
df = raw_df.rename(columns={"content": "text", "class": "label"})
print("원본 행 수:", len(df))
print("원본 라벨 분포:")
print(df["label"].value_counts(dropna=False))


원본 행 수: 19009
원본 라벨 분포:
label
2    10531
1     5778
3     2700
Name: count, dtype: int64


## 4. 문자 정제 및 라벨 충돌 검사
동일 문장에 서로 다른 라벨이 붙어 있으면 충돌 파일로 분리하고 학습에서 제외합니다.
원문과 출처 행 번호를 보존합니다. URL·숫자 마스킹은 그룹 계산에만 사용하며 모델 입력에는 적용하지 않습니다.


In [10]:
import re
import unicodedata
import pandas as pd
from datasets import load_dataset

# 0: 일반, 1: 스미싱 위험
LABEL_MAP = {
    2: 1,
    3: 0,
}

LABEL_NAMES = {
    0: "일반",
    1: "스미싱 위험",
}

def normalize_text(value):
    if not isinstance(value, str):
        return ""

    return re.sub(
        r"\s+",
        " ",
        unicodedata.normalize("NFKC", value),
    ).strip()


# 1. 데이터셋 불러오기
dataset = load_dataset(DATASET_ID)
df = dataset["train"].to_pandas()

# 2. 원본 컬럼명 변경
df = df[["content", "class"]].rename(
    columns={
        "content": "text",
        "class": "source_label",
    }
)

df["source_row"] = df.index
df["source"] = DATASET_ID
df["raw_text"] = df["text"]

# 3. 사용할 원본 라벨만 선택
clean_df = df.loc[
    df["source_label"].isin(LABEL_MAP)
].copy()

# 4. 문자 정규화 및 빈 문자 제거
clean_df["text"] = clean_df["text"].map(normalize_text)

empty_count = int(clean_df["text"].eq("").sum())

clean_df = clean_df.loc[
    clean_df["text"].ne("")
].copy()

# 5. 동일 문장에 서로 다른 라벨이 있으면 제외
label_counts = (
    clean_df.groupby("text")["source_label"]
    .nunique()
)

conflicting_texts = label_counts[
    label_counts > 1
].index

conflicts = clean_df.loc[
    clean_df["text"].isin(conflicting_texts)
].copy()

RUN_DIR.mkdir(parents=True, exist_ok=True)

conflicts.to_csv(
    RUN_DIR / "label_conflicts.csv",
    index=False,
    encoding="utf-8-sig",
)

clean_df = clean_df.loc[
    ~clean_df["text"].isin(conflicting_texts)
].copy()

# 6. 동일 문장 중복 제거
duplicate_count = int(
    clean_df.duplicated(subset=["text"]).sum()
)

clean_df = (
    clean_df.drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

# 7. 프로젝트 라벨 생성
clean_df["label"] = (
    clean_df["source_label"]
    .map(LABEL_MAP)
    .astype(int)
)

clean_df["category"] = clean_df["label"].map(LABEL_NAMES)

# 8. 검증 및 저장
assert not clean_df.empty
assert clean_df["text"].is_unique
assert set(clean_df["label"]) == {0, 1}

clean_df.to_csv(
    RUN_DIR / "clean_messages.csv",
    index=False,
    encoding="utf-8-sig",
)

print("원본 데이터:", len(df))
print("빈 문자 제외:", empty_count)
print("충돌 문장 수:", len(conflicting_texts))
print("동일 문장 중복 제외:", duplicate_count)
print("정제 후:", len(clean_df))
print()
print(clean_df["category"].value_counts())

display(
    clean_df.groupby("category", group_keys=False)
    .head(5)[
        ["text", "source_label", "label", "category"]
    ]
)

원본 데이터: 19009
빈 문자 제외: 0
충돌 문장 수: 0
동일 문장 중복 제외: 3191
정제 후: 10040

category
스미싱 위험    7631
일반        2409
Name: count, dtype: int64


,text,source_label,label,category
0,엄마 나 폰이 망가져서 수리맡기고 컴퓨터 문자나라로 메시지 보내고 있어 엄마 지금 바뻐?,2,1,스미싱 위험
1,[국외발신] [코인원] 고객님계정이 해외IP에서 로그인되였습니다.해외IP차단해주세요...,2,1,스미싱 위험
2,엄마 입금받을수있는 은행계좌번호 하나랑 계좌등록 본인인증땜에 계좌 3자리 비밀번호까...,2,1,스미싱 위험
3,엄마 바빠?나지금 핸드폰 고장나서 매장에 수리맡기고 급한대로 예전에 내명의로 가입해...,2,1,스미싱 위험
4,엄마~ 바빠? 엄마 나 급한 일이 좀 생겨서... 지금 98만원만 입금해 줄 수 있...,2,1,스미싱 위험
7631,"(우체국 배달완료) ""서로 존중, 함께 배려"" 이영희 고객님! 우체국입니다. 이씨에...",3,0,일반
7632,"(우체국 배달완료) ""서로 존중, 함께 배려"" 백서연 고객님! 우체국입니다. 이씨에...",3,0,일반
7633,"(우체국 배달완료) ""서로 존중, 함께 배려"" 김철수 고객님! 우체국입니다. 이씨에...",3,0,일반
7634,"(우체국 배달완료) ""서로 존중, 함께 배려"" 박민수 고객님! 우체국입니다. 이씨에...",3,0,일반
7635,"(우체국 배달완료) ""서로 존중, 함께 배려"" 송하나 고객님! 우체국입니다. 이씨에...",3,0,일반


In [11]:
# 원본 라벨 → 프로젝트 라벨
SOURCE_LABEL_MAP = {
    1: 0,  # 일반
    2: 1,  # 광고성
    3: 0,  # 일반
}

LABEL_NAMES = {
    0: "일반",
    1: "스미싱 위험",
}

source_labels = pd.to_numeric(
    clean_df["source_label"], errors="raise"
)

assert source_labels.isin(SOURCE_LABEL_MAP).all(), (
    "source_label에 1, 2, 3 이외의 값이 있습니다."
)

clean_df["label"] = source_labels.map(SOURCE_LABEL_MAP).astype(int)
clean_df["category"] = clean_df["label"].map(LABEL_NAMES)

clean_df["label_basis"] = "원본 라벨 기준 사용자 지정 매핑"
clean_df["reviewed"] = False

# 이전 분류에서 생성한 보조 컬럼 제거
clean_df = clean_df.drop(
    columns=["needs_review", "model_label"],
    errors="ignore",
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(
    RUN_DIR / "messages_provisional_labels.csv",
    index=False,
    encoding="utf-8-sig",
)

print("정제 후:", len(clean_df))
print(clean_df["category"].value_counts())

display(
    clean_df.groupby("label", group_keys=False)
    .head(5)[["text", "source_label", "label", "category"]]
)

정제 후: 10040
category
스미싱 위험    7631
일반        2409
Name: count, dtype: int64


,text,source_label,label,category
0,엄마 나 폰이 망가져서 수리맡기고 컴퓨터 문자나라로 메시지 보내고 있어 엄마 지금 바뻐?,2,1,스미싱 위험
1,[국외발신] [코인원] 고객님계정이 해외IP에서 로그인되였습니다.해외IP차단해주세요...,2,1,스미싱 위험
2,엄마 입금받을수있는 은행계좌번호 하나랑 계좌등록 본인인증땜에 계좌 3자리 비밀번호까...,2,1,스미싱 위험
3,엄마 바빠?나지금 핸드폰 고장나서 매장에 수리맡기고 급한대로 예전에 내명의로 가입해...,2,1,스미싱 위험
4,엄마~ 바빠? 엄마 나 급한 일이 좀 생겨서... 지금 98만원만 입금해 줄 수 있...,2,1,스미싱 위험
7631,"(우체국 배달완료) ""서로 존중, 함께 배려"" 이영희 고객님! 우체국입니다. 이씨에...",3,0,일반
7632,"(우체국 배달완료) ""서로 존중, 함께 배려"" 백서연 고객님! 우체국입니다. 이씨에...",3,0,일반
7633,"(우체국 배달완료) ""서로 존중, 함께 배려"" 김철수 고객님! 우체국입니다. 이씨에...",3,0,일반
7634,"(우체국 배달완료) ""서로 존중, 함께 배려"" 박민수 고객님! 우체국입니다. 이씨에...",3,0,일반
7635,"(우체국 배달완료) ""서로 존중, 함께 배려"" 송하나 고객님! 우체국입니다. 이씨에...",3,0,일반


## 5. 템플릿 그룹 기준 분할
URL·숫자·공백만 다른 문장을 동일 그룹으로 묶고 약 80/10/10으로 분리합니다. 그룹 크기에 따라 실제 비율은 달라집니다.
이 규칙은 의미가 같은 모든 문장을 찾아내지는 못합니다. 동일 시나리오·템플릿을 알고 있다면 출처 기반 그룹을 사용하는 것이 더 정확합니다.
검증·테스트에서 한 클래스가 빠지는 분할은 사용하지 않습니다. 그룹이 부족하면 무작위 분할로 우회하지 말고 데이터 다양성을 보강하세요.


In [12]:
import re
import hashlib
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

def template_key(text):
    key = normalize_text(text).lower()
    key = re.sub(r"(?:https?://|hxxps?://|www\.)\S+", " <URL> ", key)
    key = re.sub(r"\d+(?:[,.:/-]\d+)*", "<NUM>", key)
    key = re.sub(r"\s+", "", key)
    return hashlib.sha256(key.encode("utf-8")).hexdigest()


# 1. 라벨 확인 및 템플릿 그룹 생성
clean_df = clean_df.copy()

# 이전 코드에서 생성했을 수 있는 학습용 라벨 제거
clean_df = clean_df.drop(columns=["model_label"], errors="ignore")

assert not clean_df.empty, "정제된 데이터가 없습니다."
assert clean_df["label"].isin(LABEL_NAMES).all(), (
    "라벨은 0: 일반, 1: 광고성, 2: 스미싱 위험이어야 합니다."
)

clean_df["label"] = clean_df["label"].astype(int)
clean_df["category"] = clean_df["label"].map(LABEL_NAMES)
clean_df["group_id"] = clean_df["text"].map(template_key)

# 실제 존재하는 클래스가 각 분할에 포함되도록 검사
expected_labels = set(clean_df["label"].unique())
assert len(expected_labels) >= 2, "최소 두 개 클래스가 필요합니다."

print("클래스별 그룹 수:")
group_counts = clean_df.groupby("label")["group_id"].nunique()
print(group_counts.rename(index=LABEL_NAMES))

missing_labels = set(LABEL_NAMES) - expected_labels
if missing_labels:
    print(
        "현재 데이터에 없는 클래스:",
        [LABEL_NAMES[label] for label in sorted(missing_labels)],
    )

assert group_counts.min() >= 3, (
    "학습·검증·테스트용 그룹이 부족합니다."
)

# 2. 학습 약 80%, 나머지 약 20% 분할
outer = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

chosen = None

for train_idx, rest_idx in outer.split(
    clean_df,
    clean_df["label"],
    clean_df["group_id"],
):
    train_part = clean_df.iloc[train_idx]
    rest = clean_df.iloc[rest_idx]

    if (
        set(train_part["label"]) != expected_labels
        or set(rest["label"]) != expected_labels
    ):
        continue

    if rest.groupby("label")["group_id"].nunique().min() < 2:
        continue

    # 3. 검증 약 10%, 테스트 약 10% 분할
    inner = StratifiedGroupKFold(
        n_splits=2,
        shuffle=True,
        random_state=SEED,
    )

    for valid_idx, test_idx in inner.split(
        rest,
        rest["label"],
        rest["group_id"],
    ):
        valid_part = rest.iloc[valid_idx]
        test_part = rest.iloc[test_idx]

        if (
            set(valid_part["label"]) == expected_labels
            and set(test_part["label"]) == expected_labels
        ):
            chosen = (train_part, valid_part, test_part)
            break

    if chosen is not None:
        break

assert chosen is not None, (
    "모든 현재 클래스가 포함된 그룹 분할을 만들지 못했습니다. "
    "템플릿별 클래스 분포를 확인하세요."
)

train_df, valid_df, test_df = [
    part.reset_index(drop=True).copy()
    for part in chosen
]

splits = {
    "train": train_df,
    "valid": valid_df,
    "test": test_df,
}

# 4. 동일 문장 및 동일 그룹의 분할 간 중복 검사
for left, right in [
    ("train", "valid"),
    ("train", "test"),
    ("valid", "test"),
]:
    assert set(splits[left]["text"]).isdisjoint(
        splits[right]["text"]
    ), f"{left}와 {right}에 동일 문장이 있습니다."

    assert set(splits[left]["group_id"]).isdisjoint(
        splits[right]["group_id"]
    ), f"{left}와 {right}에 동일 그룹이 있습니다."

assert sum(len(frame) for frame in splits.values()) == len(clean_df)

# 5. 저장 및 결과 확인
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

split_summary = {}

for name, frame in splits.items():
    split_summary[name] = {
        "문장 수": len(frame),
        "비율": round(len(frame) / len(clean_df), 3),
        "그룹 수": int(frame["group_id"].nunique()),
        **{
            label_name: int(frame["label"].eq(label).sum())
            for label, label_name in LABEL_NAMES.items()
        },
    }

    frame.to_csv(
        SPLIT_DIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig",
    )

display(pd.DataFrame(split_summary).T)

클래스별 그룹 수:
label
일반         830
스미싱 위험    7311
Name: group_id, dtype: int64


,문장 수,비율,그룹 수,일반,스미싱 위험
train,7944.0,0.791,6518.0,1852.0,6092.0
valid,1101.0,0.110,818.0,306.0,795.0
test,995.0,0.099,805.0,251.0,744.0


## 6. 토크나이저·모델 로드 및 토큰화
모델 로드는 한 번만 실행합니다. 256 토큰을 초과하는 입력은 잘리므로 분할별 비율도 기록합니다.


In [13]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, num_labels=2,
    id2label=ID2LABEL, label2id={v: k for k, v in ID2LABEL.items()},
)
raw_datasets = DatasetDict({
    name: Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    for name, frame in splits.items()
})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = raw_datasets.map(tokenize, batched=True, remove_columns=["text"])
tokenized = tokenized.rename_column("label", "labels")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
truncation_rates = {}
for name, frame in splits.items():
    lengths = tokenizer(frame["text"].tolist(), truncation=False, return_length=True)["length"]
    truncation_rates[name] = float(np.mean(np.array(lengths) > MAX_LENGTH))
print("분할별 잘리는 입력 비율:", truncation_rates)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/7944 [00:00<?, ? examples/s]

Map:   0%|          | 0/1101 [00:00<?, ? examples/s]

Map:   0%|          | 0/995 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (603 > 512). Running this sequence through the model will result in indexing errors


분할별 잘리는 입력 비율: {'train': 0.01787512588116818, 'valid': 0.014532243415077202, 'test': 0.018090452261306532}


## 7. 평가 지표와 학습
검증 Macro F1으로 최고 체크포인트를 선택합니다. 동점이면 첫 최고 체크포인트가 유지될 수 있습니다.
정확도뿐 아니라 위험 Recall과 정상 오탐률도 확인하세요. 테스트 세트로 설정을 조정하지 마세요.


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from transformers import TrainingArguments, Trainer

def compute_metrics(prediction):
    labels = prediction.label_ids
    predicted = np.argmax(prediction.predictions, axis=-1)

    tn, fp, fn, tp = confusion_matrix(
        labels, predicted, labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(
            labels, predicted, average="macro", zero_division=0
        ),
        "risk_precision": precision_score(
            labels, predicted, pos_label=1, zero_division=0
        ),
        "risk_recall": recall_score(
            labels, predicted, pos_label=1, zero_division=0
        ),
        "normal_false_positive_rate": fp / max(tn + fp, 1),
    }

training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["valid"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 학습 중단 시에도 출처와 분할 정의를 확인할 수 있게 먼저 저장
manifest = {
    "run_id": RUN_ID,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "dataset_fingerprint": dataset["train"]._fingerprint,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "source_label_map": LABEL_MAP,
    "id2label": ID2LABEL,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "split_method": "StratifiedGroupKFold 5 folds, then 2 folds; URL/number template groups",
    "split_summary": split_summary,
    "truncation_rates": truncation_rates,
    "environment": environment,
    "gpu": torch.cuda.get_device_name(0),
    "limitations": ["spam and smishing share risk label", "normal texts have limited variety", "template grouping is heuristic"],
}
with open(RUN_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
with open(RUN_DIR / "training_args.json", "w", encoding="utf-8") as f:
    f.write(training_args.to_json_string())

trainer.train()
validation_metrics = trainer.evaluate()
print(validation_metrics)
print("선택된 체크포인트:", trainer.state.best_model_checkpoint)
print("최고 검증 점수:", trainer.state.best_metric)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Risk Precision,Risk Recall,Normal False Positive Rate
1,0.002000,0.000891,1.000000,1.000000,1.000000,1.000000,0.000000
2,0.000700,0.000345,1.000000,1.000000,1.000000,1.000000,0.000000
3,0.000400,0.000251,1.000000,1.000000,1.000000,1.000000,0.000000


{'eval_loss': 0.0008907478186301887, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0, 'eval_risk_precision': 1.0, 'eval_risk_recall': 1.0, 'eval_normal_false_positive_rate': 0.0, 'eval_runtime': 1.1212, 'eval_samples_per_second': 981.943, 'eval_steps_per_second': 31.215, 'epoch': 3.0}
선택된 체크포인트: C:\Users\SSAFY\smishing-checker\experiments\koelectra-baseline-v2-20260907T005040704006Z\checkpoints\checkpoint-497
최고 검증 점수: 1.0


## 8. 테스트 평가와 모델 저장
학습 설정을 결정한 뒤 한 번 평가합니다. 모델·토크나이저·분할·설정·문장별 예측을 실험 폴더에 함께 보관합니다.
위험 점수는 보정된 실제 스미싱 확률이 아닙니다. NORMAL/RISK는 이 실험의 임시 분류 라벨입니다.


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

# 모델 저장 폴더
save_dir = MODEL_DIR
save_dir.mkdir(parents=True, exist_ok=True)

# 테스트 데이터 예측
prediction = trainer.predict(tokenized["test"])
predicted = prediction.predictions.argmax(axis=-1)

risk_scores = torch.softmax(
    torch.as_tensor(prediction.predictions),
    dim=-1,
)[:, 1].numpy()

# 평가 결과
report = classification_report(
    prediction.label_ids,
    predicted,
    labels=[0, 1],
    target_names=["정상", "위험"],
    output_dict=True,
    zero_division=0,
)

print(
    classification_report(
        prediction.label_ids,
        predicted,
        labels=[0, 1],
        target_names=["정상", "위험"],
        digits=4,
        zero_division=0,
    )
)

matrix = confusion_matrix(
    prediction.label_ids,
    predicted,
    labels=[0, 1],
)

display(
    pd.DataFrame(
        matrix,
        index=["실제 정상", "실제 위험"],
        columns=["예측 정상", "예측 위험"],
    )
)

# 테스트 예측 결과 저장
test_results = test_df.copy()

assert np.array_equal(
    test_results["label"].to_numpy(),
    prediction.label_ids,
), "test_df와 tokenized['test']의 데이터 순서가 다릅니다."

test_results["predicted_label"] = predicted
test_results["risk_score"] = risk_scores
test_results["correct"] = (
    test_results["label"]
    == test_results["predicted_label"]
)

test_results.to_csv(
    RUN_DIR / "test_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

# 모델과 토크나이저 저장
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

trainer.state.save_to_json(
    str(RUN_DIR / "trainer_state.json")
)

# validation_metrics가 없으면 검증 데이터 평가
if "validation_metrics" not in globals():
    validation_metrics = trainer.evaluate(
        tokenized["valid"],
        metric_key_prefix="validation",
    )

# 평가 결과 JSON 저장
result_files = {
    "test_metrics.json": prediction.metrics,
    "validation_metrics.json": validation_metrics,
    "classification_report.json": report,
    "confusion_matrix.json": {
        "label_order": [0, 1],
        "label_names": ["NORMAL", "RISK"],
        "matrix": matrix.tolist(),
    },
}

for filename, value in result_files.items():
    with open(
        RUN_DIR / filename,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            value,
            file,
            ensure_ascii=False,
            indent=2,
        )

# manifest가 없으면 새로 생성
if "manifest" not in globals():
    manifest = {}

manifest["best_checkpoint"] = (
    trainer.state.best_model_checkpoint
)
manifest["best_metric"] = trainer.state.best_metric
manifest["saved_model_dir"] = str(save_dir.resolve())

# 실제 저장된 분할 파일의 해시 기록
manifest["split_sha256"] = {}

for name in ["train", "valid", "test"]:
    split_path = SPLIT_DIR / f"{name}.csv"

    if split_path.exists():
        manifest["split_sha256"][name] = hashlib.sha256(
            split_path.read_bytes()
        ).hexdigest()

with open(
    RUN_DIR / "manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("모델 저장 위치:", save_dir.resolve())
print("실험 결과 위치:", RUN_DIR.resolve())

              precision    recall  f1-score   support

          정상     1.0000    1.0000    1.0000       251
          위험     1.0000    1.0000    1.0000       744

    accuracy                         1.0000       995
   macro avg     1.0000    1.0000    1.0000       995
weighted avg     1.0000    1.0000    1.0000       995



,예측 정상,예측 위험
실제 정상,251,0
실제 위험,0,744


모델 저장 위치: C:\Users\SSAFY\smishing-checker\experiments\koelectra-baseline-v2-20260907T005040704006Z\model
실험 결과 위치: C:\Users\SSAFY\smishing-checker\experiments\koelectra-baseline-v2-20260907T005040704006Z


## 학습 노트북 종료 — 여기까지 실행

다음 작업은 다양한 정상 문자 보강, 별도 합성 문자 평가, 저장 모델을 다시 불러온 추론 검증, OCR 연동 순서입니다.
새 사건 데이터의 평가 매핑은 요청대로 `benign=0`, `benign_lookalike=1`, `malicious=1`을 유지하되 사건 라벨을 문자 라벨로 간주한 평가임을 표시하세요.

**아래는 이전 코드 보관 영역입니다. 이번 수정 범위에서 제외했으며 그대로 실행하지 마세요.** 이전 추론 셀은 기존 v1 모델 경로를 사용하므로 이번 실험을 평가하려면 `MODEL_DIR`을 위에 출력된 `save_dir` 경로로 변경해야 합니다.


In [34]:
from pathlib import Path
import string

project_candidates = []

for letter in string.ascii_uppercase:
    drive = Path(f"{letter}:/")

    if not drive.exists():
        continue

    possible_paths = [
        drive / "smishing-checker",
        drive / "My Drive" / "smishing-checker",
        drive / "내 드라이브" / "smishing-checker",
        drive / "Google Drive" / "smishing-checker",
    ]

    for path in possible_paths:
        print(
            "확인:",
            path,
            "→",
            "존재함" if path.exists() else "없음",
        )

        if path.is_dir():
            project_candidates.append(path)

if not project_candidates:
    raise FileNotFoundError(
        "Google Drive에서 smishing-checker 폴더를 찾지 못했습니다. "
        "탐색기에서 폴더의 실제 위치를 확인하세요."
    )

drive_root = project_candidates[0]

print("\n사용할 경로:", drive_root.resolve())

incidents_candidates = [
    path
    for path in drive_root.rglob("incidents")
    if path.is_dir()
]

if not incidents_candidates:
    print("incidents 폴더가 없습니다.")
else:
    for path in incidents_candidates:
        json_files = list(path.rglob("*.json"))

        print("\nincidents 경로:", path.resolve())
        print("JSON 개수:", len(json_files))

확인: C:\smishing-checker → 없음
확인: C:\My Drive\smishing-checker → 없음
확인: C:\내 드라이브\smishing-checker → 없음
확인: C:\Google Drive\smishing-checker → 없음


FileNotFoundError: Google Drive에서 smishing-checker 폴더를 찾지 못했습니다. 탐색기에서 폴더의 실제 위치를 확인하세요.

In [27]:
from pathlib import Path

# 실제 Google Drive 경로에 맞게 수정
drive_root = Path(
    r"G:\내 드라이브\smishing-checker"
)

assert drive_root.exists(), (
    f"Google Drive 경로가 없습니다: {drive_root}"
)

candidates = [
    path
    for path in drive_root.rglob("incidents")
    if path.is_dir()
]

if not candidates:
    print("incidents 폴더를 찾지 못했습니다.")
    print("검색 기준:", drive_root.resolve())
else:
    total_json_count = 0

    for path in candidates:
        json_files = list(path.rglob("*.json"))
        total_json_count += len(json_files)

        print("\n폴더 경로:", path.resolve())
        print("JSON 개수:", len(json_files))

    print("\n전체 JSON 개수:", total_json_count)

AssertionError: Google Drive 경로가 없습니다: G:\내 드라이브\smishing-checker

In [1]:
from pathlib import Path
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# 1. 저장된 모델 경로
PROJECT_DIR = Path("/content/drive/MyDrive/smishing-checker")
MODEL_DIR = PROJECT_DIR / "models/koelectra-baseline-v1"
OUTPUT_DIR = PROJECT_DIR / "evaluation"

assert MODEL_DIR.exists(), f"모델 폴더가 없습니다: {MODEL_DIR}"
assert not new_df.empty, "먼저 JSON 데이터 읽기 셀을 실행하세요."

# 2. 모델 입력 준비 — 원문 text는 그대로 보존
result_df = new_df.copy()
result_df["model_text"] = (
    result_df["text"]
    .str.replace(r"^\s*\[TRAINING\]\s*", "", regex=True)
    .str.strip()
)

assert result_df["model_text"].notna().all(), "문자 내용이 누락됐습니다."
assert result_df["model_text"].str.len().gt(0).all(), "빈 문자가 있습니다."

# 3. 저장된 토크나이저와 모델 불러오기
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

eval_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
eval_model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
).to(device)
eval_model.eval()

# 앞서 학습한 모델의 라벨 순서 확인
assert eval_model.config.id2label == {0: "NORMAL", 1: "RISK"}, (
    f"모델 라벨 확인 필요: {eval_model.config.id2label}"
)

# 4. 16개씩 배치 예측
texts = result_df["model_text"].tolist()
risk_scores = []

with torch.inference_mode():
    for start in range(0, len(texts), 16):
        inputs = eval_tokenizer(
            texts[start:start + 16],
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        ).to(device)

        logits = eval_model(**inputs).logits
        scores = torch.softmax(logits, dim=-1)[:, 1]
        risk_scores.extend(scores.cpu().tolist())

# 5. 예측 결과 기록
result_df["risk_score"] = risk_scores
result_df["predicted_label"] = (
    result_df["risk_score"] >= 0.5
).astype(int)

result_df["prediction"] = result_df["predicted_label"].map({
    0: "정상",
    1: "위험",
})

print(f"\n예측 완료: {len(result_df)}개 문자")
print(result_df["prediction"].value_counts())

display(result_df[
    ["incident_id", "case_type", "model_text", "prediction", "risk_score"]
].head(20))

AssertionError: 모델 폴더가 없습니다: /content/drive/MyDrive/smishing-checker/models/koelectra-baseline-v1

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

# 이번 평가에 적용할 라벨 매핑
label_map = {
    "benign": 0,
    "benign_lookalike": 1,
    "malicious": 1,
}

scored_df = result_df.copy()

unknown = set(scored_df["case_type"].dropna()) - set(label_map)
assert not unknown, f"정의하지 않은 라벨: {unknown}"
assert scored_df["case_type"].notna().all(), "사건 라벨 누락이 있습니다."

scored_df["expected_label"] = (
    scored_df["case_type"].map(label_map).astype(int)
)
scored_df["predicted_label"] = (
    scored_df["risk_score"] >= 0.5
).astype(int)
scored_df["correct"] = (
    scored_df["expected_label"] == scored_df["predicted_label"]
)

# 평가 결과
print(f"평가 문자 수: {len(scored_df)}")
print(f"정확도: {accuracy_score(
    scored_df['expected_label'],
    scored_df['predicted_label']
):.2%}")

print(classification_report(
    scored_df["expected_label"],
    scored_df["predicted_label"],
    labels=[0, 1],
    target_names=["정상", "위험"],
    digits=4,
    zero_division=0,
))

matrix = confusion_matrix(
    scored_df["expected_label"],
    scored_df["predicted_label"],
    labels=[0, 1],
)

print("혼동행렬: 행=정답, 열=예측")
display(pd.DataFrame(
    matrix,
    index=["실제 정상", "실제 위험"],
    columns=["예측 정상", "예측 위험"],
))

# 오분류 확인
mistakes = scored_df.loc[~scored_df["correct"]]

print(f"\n오분류: {len(mistakes)}개")
display(mistakes[
    ["incident_id", "case_type", "model_text", "prediction", "risk_score"]
])

# 기존 검수 파일과 별도로 저장
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y%m%d-%H%M%S-%f")
score_path = OUTPUT_DIR / f"koelectra-case-label-eval-{timestamp}.csv"

scored_df.to_csv(score_path, index=False, encoding="utf-8-sig")
print("저장 위치:", score_path)

평가 문자 수: 104
정확도: 70.19%
              precision    recall  f1-score   support

          정상     0.0000    0.0000    0.0000        31
          위험     0.7019    1.0000    0.8249        73

    accuracy                         0.7019       104
   macro avg     0.3510    0.5000    0.4124       104
weighted avg     0.4927    0.7019    0.5790       104

혼동행렬: 행=정답, 열=예측


,예측 정상,예측 위험
실제 정상,0,31
실제 위험,0,73



오분류: 31개


,incident_id,case_type,model_text,prediction,risk_score
0,inc-MOB-BEN-005,benign,"고객님, 사용 중이신 금융 서비스의 보안 정책이 강화되었습니다. 원활한 서비스 이용...",위험,0.999310
1,inc-MOB-BEN-006,benign,[정부24] 미수령 환급금 안내. 귀하의 미수령 환급금이 발생하였습니다. 기한 내 ...,위험,0.999362
2,inc-MOB-BEN-007,benign,엄마 나 휴대폰 액정이 깨져서 수리 맡겼어. 지금 급하게 송금할 곳이 있는데 인증이...,위험,0.999291
6,inc-MOB-BEN-015,benign,"[정부24] 고객님, 미수령 정부지원환급금 482,000원이 확인되었습니다. 금일 ...",위험,0.999358
7,inc-MOB-BEN-016,benign,[계정안내] 고객님의 계정에 새로운 기기(Windows/인천)에서 로그인이 시도되었...,위험,0.999341
8,inc-MOB-BEN-017,benign,[보안알림] 고객님의 계정에 새로운 기기(iPhone 15)에서 로그인이 감지되었습...,위험,0.999323
12,inc-MOB-BEN-025,benign,[금융안내] 고객님의 공동인증서 만료가 3일 남았습니다. 만료 시 모든 금융거래가 ...,위험,0.999238
13,inc-MOB-BEN-026,benign,"[알림] 고객님, 이번 달 데이터 기본 제공량이 모두 소진되었습니다. 현재 기준 초...",위험,0.999185
14,inc-MOB-BEN-027,benign,2024년도 미수령 정부지원금(환급금)이 존재합니다. 신청 기한이 임박하였으니 아래...,위험,0.999357
18,inc-MOB-BEN-035,benign,"[Web발급] 07/24 12:35 카드 승인 458,000원(일시불) 본인 요청 ...",위험,0.999384


저장 위치: /content/drive/MyDrive/smishing-checker/evaluation/koelectra-case-label-eval-20260906-212737-702773.csv


# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# ============================================================
# 커스텀 Dataset
# ============================================================
class VQAMCDataset(Dataset):
    def __init__(
        self,
        df,
        processor,
        train=True,
    ):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image = Image.open(
            row["path"]
        ).convert("RGB")

        question = str(row["question"])

        choice_a = str(row["a"])
        choice_b = str(row["b"])
        choice_c = str(row["c"])
        choice_d = str(row["d"])

        user_text = build_mc_prompt(
            question,
            choice_a,
            choice_b,
            choice_c,
            choice_d,
        )

        messages = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": SYSTEM_INSTRUCT,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image,
                    },
                    {
                        "type": "text",
                        "text": user_text,
                    },
                ],
            },
        ]

        sample = {
            "messages": messages,
            "image": image,
        }

        if self.train:
            gold = (
                str(row["answer"])
                .strip()
                .lower()
            )

            if gold not in {
                "a",
                "b",
                "c",
                "d",
            }:
                raise ValueError(
                    f"잘못된 정답입니다: {gold}"
                )

            messages.append({
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": gold,
                    }
                ],
            })

            # DataCollator가 정답 위치를 찾을 수 있도록 저장
            sample["gold"] = gold

        return sample


# ============================================================
# Data Collator
# ============================================================
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts = []
        images = []

        for sample in batch:
            messages = sample["messages"]
            image = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,

                # 학습: assistant 답이 이미 포함되어 있음
                # 추론: assistant 답변 시작 위치 추가
                add_generation_prompt=not self.train,
            )

            texts.append(text)
            images.append(image)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        # ====================================================
        # 학습일 때 정답 토큰에만 Loss 계산
        # ====================================================
        if self.train:
            # 전체를 우선 -100으로 마스킹
            labels = torch.full_like(
                enc["input_ids"],
                fill_value=-100,
            )

            for batch_index, sample in enumerate(batch):
                gold = sample["gold"]

                answer_token_ids = (
                    self.processor
                    .tokenizer
                    .encode(
                        gold,
                        add_special_tokens=False,
                    )
                )

                if len(answer_token_ids) == 0:
                    raise RuntimeError(
                        f"정답 토큰화 실패: {gold}"
                    )

                input_ids = enc["input_ids"][
                    batch_index
                ]

                answer_length = len(
                    answer_token_ids
                )

                answer_start = None

                # Assistant 정답은 입력 끝쪽에 있으므로
                # 뒤에서부터 마지막 출현 위치를 검색
                for start in range(
                    len(input_ids) - answer_length,
                    -1,
                    -1,
                ):
                    candidate = input_ids[
                        start:
                        start + answer_length
                    ].tolist()

                    if candidate == answer_token_ids:
                        answer_start = start
                        break

                if answer_start is None:
                    decoded_input = (
                        self.processor
                        .tokenizer
                        .decode(
                            input_ids.tolist(),
                            skip_special_tokens=False,
                        )
                    )

                    raise RuntimeError(
                        "입력에서 정답 토큰을 찾지 못했습니다.\n"
                        f"정답: {gold}\n"
                        f"정답 토큰: {answer_token_ids}\n"
                        f"입력: {decoded_input}"
                    )

                answer_end = (
                    answer_start +
                    answer_length
                )

                # 정답 부분에만 실제 token ID를 입력
                labels[
                    batch_index,
                    answer_start:answer_end,
                ] = input_ids[
                    answer_start:answer_end
                ]

            enc["labels"] = labels

        return enc

In [ ]:
# ============================================================
# Dataset 생성
# ============================================================
train_ds = VQAMCDataset(
    df=train_subset,
    processor=processor,
    train=True,
)

valid_ds = VQAMCDataset(
    df=valid_subset,
    processor=processor,
    train=True,
)


# ============================================================
# DataLoader 생성
# ============================================================
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(
        processor=processor,
        train=True,
    ),
    num_workers=0,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(
        processor=processor,
        train=True,
    ),
    num_workers=0,
)


# ============================================================
# 생성 결과 확인
# ============================================================
print("Train Dataset:", len(train_ds))
print("Valid Dataset:", len(valid_ds))

print("Train Loader batches:", len(train_loader))
print("Valid Loader batches:", len(valid_loader))

In [ ]:
check_batch = next(iter(train_loader))

print(
    "input_ids shape:",
    check_batch["input_ids"].shape,
)

print(
    "labels shape:",
    check_batch["labels"].shape,
)

for batch_index in range(
    check_batch["labels"].size(0)
):
    labels = check_batch["labels"][
        batch_index
    ]

    supervised_tokens = labels[
        labels != -100
    ]

    supervised_text = (
        processor
        .tokenizer
        .decode(
            supervised_tokens.tolist(),
            skip_special_tokens=False,
        )
    )

    print(
        f"샘플 {batch_index} Loss 대상:",
        repr(supervised_text),
    )

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# ============================================================
# 기존 train_subset, valid_subset 사용
# 다시 분할하지 않음
# ============================================================

print("Train subset:", len(train_subset))
print("Valid subset:", len(valid_subset))


# ============================================================
# Dataset 생성
# ============================================================
train_ds = VQAMCDataset(
    df=train_subset,
    processor=processor,
    train=True,
)

valid_ds = VQAMCDataset(
    df=valid_subset,
    processor=processor,
    train=True,
)


# ============================================================
# DataLoader 생성
# ============================================================
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(
        processor=processor,
        train=True,
    ),
    num_workers=0,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(
        processor=processor,
        train=True,
    ),
    num_workers=0,
)


# ============================================================
# 결과 확인
# ============================================================
print("Train Dataset:", len(train_ds))
print("Valid Dataset:", len(valid_ds))

print("Train Loader batches:", len(train_loader))
print("Valid Loader batches:", len(valid_loader))

In [ ]:
check_batch = next(iter(train_loader))

labels = check_batch["labels"][0]

supervised_tokens = labels[
    labels != -100
]

supervised_text = (
    processor
    .tokenizer
    .decode(
        supervised_tokens.tolist(),
        skip_special_tokens=False,
    )
)

print(
    "Loss 대상:",
    repr(supervised_text),
)

In [ ]:
import os
import math
import torch

from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup


# ============================================================
# 1. 학습 설정
# ============================================================
# 처음 작동 확인은 1, 최종 학습은 3
EPOCHS = 1

GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY = 0.01

SAVE_DIR = os.path.join(
    BASE_PATH,
    "qwen3_vl_4b_lora_best",
)


# ============================================================
# 2. 실행 전 객체 확인
# ============================================================
required_objects = [
    "model",
    "processor",
    "train_loader",
    "valid_loader",
    "compute_dtype",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "학습 전에 다음 객체를 생성해야 합니다: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 3. 모델 디바이스 확인
# ============================================================
# device_map="auto"를 사용했으므로 model.to()를 호출하지 않음
model_device = (
    model
    .get_input_embeddings()
    .weight
    .device
)

print("Model device:", model_device)
print("Compute dtype:", compute_dtype)


# Gradient checkpointing 사용 시 cache 비활성화
if hasattr(model, "config"):
    model.config.use_cache = False


# ============================================================
# 4. LoRA 학습 파라미터만 선택
# ============================================================
trainable_params = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

trainable_count = sum(
    parameter.numel()
    for parameter in trainable_params
)

print(
    "Trainable parameters:",
    f"{trainable_count:,}",
)


# ============================================================
# 5. Optimizer
# ============================================================
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# 6. Scheduler
# ============================================================
optimizer_steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

num_training_steps = (
    EPOCHS * optimizer_steps_per_epoch
)

num_warmup_steps = max(
    1,
    int(num_training_steps * 0.03),
)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print(
    "Optimizer steps per epoch:",
    optimizer_steps_per_epoch,
)
print(
    "Total optimizer steps:",
    num_training_steps,
)
print(
    "Warmup steps:",
    num_warmup_steps,
)


# ============================================================
# 7. GradScaler 설정
# ============================================================
# FP16에서만 활성화하고 BF16에서는 비활성화
use_grad_scaler = (
    compute_dtype == torch.float16
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_grad_scaler,
)

print("GradScaler enabled:", use_grad_scaler)


# ============================================================
# 8. a/b/c/d 토큰 준비
# ============================================================
choices = ["a", "b", "c", "d"]
choice_token_ids = []

for choice in choices:
    token_ids = processor.tokenizer.encode(
        choice,
        add_special_tokens=False,
    )

    if len(token_ids) != 1:
        raise RuntimeError(
            f"선택지 {choice}가 한 토큰이 아닙니다: "
            f"{token_ids}"
        )

    choice_token_ids.append(
        token_ids[0]
    )

choice_token_ids_tensor = torch.tensor(
    choice_token_ids,
    device=model_device,
)

choice_id_to_index = {
    token_id: index
    for index, token_id in enumerate(
        choice_token_ids
    )
}

print(
    "Choice token IDs:",
    dict(zip(choices, choice_token_ids)),
)


# ============================================================
# 9. 학습 준비
# ============================================================
best_val_accuracy = -1.0
best_epoch = -1
global_step = 0

optimizer.zero_grad(set_to_none=True)

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)


# ============================================================
# 10. 학습 루프
# ============================================================
for epoch in range(EPOCHS):

    # ========================================================
    # 10-1. Train
    # ========================================================
    model.train()

    train_loss_sum = 0.0
    train_batch_count = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS} [train]",
        unit="batch",
    )

    for step, batch in enumerate(
        train_bar,
        start=1,
    ):
        batch = {
            key: value.to(model_device)
            for key, value in batch.items()
            if torch.is_tensor(value)
        }

        # 마지막 accumulation 묶음이 4개보다 적은 경우 처리
        group_start = (
            ((step - 1) // GRAD_ACCUM)
            * GRAD_ACCUM
        )

        current_accumulation_size = min(
            GRAD_ACCUM,
            len(train_loader) - group_start,
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=compute_dtype,
        ):
            outputs = model(**batch)
            raw_loss = outputs.loss

            loss = (
                raw_loss /
                current_accumulation_size
            )

        if not torch.isfinite(raw_loss):
            raise RuntimeError(
                "정상적이지 않은 loss가 발생했습니다. "
                f"epoch={epoch + 1}, "
                f"step={step}, "
                f"loss={raw_loss.item()}"
            )

        # BF16에서는 일반 backward,
        # FP16에서는 GradScaler가 자동 적용
        scaler.scale(loss).backward()

        train_loss_sum += (
            raw_loss
            .detach()
            .float()
            .item()
        )
        train_batch_count += 1

        should_update = (
            step % GRAD_ACCUM == 0
            or step == len(train_loader)
        )

        if should_update:
            # FP16 사용 시 gradient를 원래 크기로 복구
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                max_norm=MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

        average_train_loss = (
            train_loss_sum /
            train_batch_count
        )

        train_bar.set_postfix({
            "loss": f"{average_train_loss:.4f}",
            "lr": (
                f"{scheduler.get_last_lr()[0]:.2e}"
            ),
        })


    # ========================================================
    # 10-2. Validation
    # ========================================================
    model.eval()

    val_loss_sum = 0.0
    val_batch_count = 0

    correct = 0
    total = 0
    skipped = 0

    with torch.no_grad():
        valid_bar = tqdm(
            valid_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS} [valid]",
            unit="batch",
        )

        for batch in valid_bar:
            batch = {
                key: value.to(model_device)
                for key, value in batch.items()
                if torch.is_tensor(value)
            }

            with torch.amp.autocast(
                device_type="cuda",
                dtype=compute_dtype,
            ):
                outputs = model(**batch)
                val_batch_loss = outputs.loss

            val_loss_sum += (
                val_batch_loss
                .detach()
                .float()
                .item()
            )
            val_batch_count += 1

            logits = outputs.logits
            labels = batch["labels"]

            batch_size = labels.size(0)

            for sample_index in range(batch_size):
                sample_labels = labels[
                    sample_index
                ]

                # 정답만 Loss 대상으로 만들었으므로
                # -100이 아닌 위치가 정답 위치
                answer_positions = torch.nonzero(
                    sample_labels != -100,
                    as_tuple=False,
                ).flatten()

                if answer_positions.numel() == 0:
                    skipped += 1
                    continue

                answer_position = (
                    answer_positions[0].item()
                )

                if answer_position == 0:
                    skipped += 1
                    continue

                gold_token_id = (
                    sample_labels[
                        answer_position
                    ].item()
                )

                if gold_token_id not in choice_id_to_index:
                    skipped += 1
                    continue

                gold_choice_index = (
                    choice_id_to_index[
                        gold_token_id
                    ]
                )

                # 정답 토큰은 바로 이전 위치 logits에서 예측
                next_token_logits = logits[
                    sample_index,
                    answer_position - 1,
                ]

                # 최종 추론과 동일하게 a/b/c/d만 비교
                choice_logits = next_token_logits[
                    choice_token_ids_tensor
                ]

                predicted_choice_index = (
                    choice_logits
                    .float()
                    .argmax(dim=-1)
                    .item()
                )

                if (
                    predicted_choice_index
                    == gold_choice_index
                ):
                    correct += 1

                total += 1

            current_accuracy = (
                correct / total
                if total > 0
                else 0.0
            )

            valid_bar.set_postfix({
                "loss": (
                    f"{val_loss_sum / val_batch_count:.4f}"
                ),
                "accuracy": (
                    f"{current_accuracy:.4f}"
                ),
            })


    # ========================================================
    # 10-3. Epoch 결과
    # ========================================================
    average_train_loss = (
        train_loss_sum /
        max(train_batch_count, 1)
    )

    average_val_loss = (
        val_loss_sum /
        max(val_batch_count, 1)
    )

    val_accuracy = (
        correct / total
        if total > 0
        else 0.0
    )

    print(
        f"\n[Epoch {epoch + 1}/{EPOCHS}] "
        f"train loss={average_train_loss:.4f}, "
        f"valid loss={average_val_loss:.4f}, "
        f"valid accuracy={val_accuracy:.4f} "
        f"({correct}/{total}), "
        f"skipped={skipped}, "
        f"optimizer steps={global_step}"
    )

    if total == 0:
        raise RuntimeError(
            "Validation 정답을 하나도 계산하지 못했습니다. "
            "DataCollator의 Loss Masking을 확인하세요."
        )


    # ========================================================
    # 10-4. 최고 정확도 모델 저장
    # ========================================================
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1

        model.save_pretrained(
            SAVE_DIR
        )

        processor.save_pretrained(
            SAVE_DIR
        )

        print(
            "Best checkpoint 저장: "
            f"epoch={best_epoch}, "
            f"accuracy={best_val_accuracy:.4f}, "
            f"path={SAVE_DIR}"
        )

    else:
        print(
            "Checkpoint 저장하지 않음: "
            f"현재 accuracy={val_accuracy:.4f}, "
            f"최고 accuracy={best_val_accuracy:.4f}"
        )


# ============================================================
# 11. 학습 종료
# ============================================================
if hasattr(model, "config"):
    model.config.use_cache = True

print("\n학습 완료")
print("Best epoch:", best_epoch)
print(
    "Best validation accuracy:",
    f"{best_val_accuracy:.4f}",
)
print("Best model path:", SAVE_DIR)

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

In [ ]:
import os

local_path = os.path.abspath("submission.csv")

print("파일 위치:", local_path)
print("파일 존재:", os.path.exists(local_path))
print("파일 크기:", os.path.getsize(local_path), "bytes")

In [ ]:
from google.colab import files

files.download(local_path)

In [ ]:
# 모델 응답 예시
print(output_text)